# Очистка данных

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
os.chdir('..')

## Загрузка данных

In [3]:
sales = pd.read_csv("data/raw/sales_train.csv")
item_categories = pd.read_csv("data/raw/item_categories.csv")
items = pd.read_csv("data/raw/items.csv")
shops = pd.read_csv("data/raw/shops.csv")


In [4]:
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 2935849 entries, 0 to 2935848
Data columns (total 6 columns):
 #   Column          Dtype  
---  ------          -----  
 0   date            str    
 1   date_block_num  int64  
 2   shop_id         int64  
 3   item_id         int64  
 4   item_price      float64
 5   item_cnt_day    float64
dtypes: float64(2), int64(3), str(1)
memory usage: 162.4 MB


In [5]:
item_categories.info()

<class 'pandas.DataFrame'>
RangeIndex: 84 entries, 0 to 83
Data columns (total 2 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   item_category_name  84 non-null     str  
 1   item_category_id    84 non-null     int64
dtypes: int64(1), str(1)
memory usage: 4.6 KB


In [6]:
items.info()

<class 'pandas.DataFrame'>
RangeIndex: 22170 entries, 0 to 22169
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   item_name         22170 non-null  str  
 1   item_id           22170 non-null  int64
 2   item_category_id  22170 non-null  int64
dtypes: int64(2), str(1)
memory usage: 1.8 MB


In [7]:
shops.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   shop_name  60 non-null     str  
 1   shop_id    60 non-null     int64
dtypes: int64(1), str(1)
memory usage: 3.5 KB


In [8]:
sales = sales.groupby(['date', 'date_block_num', 'shop_id', 'item_id'], as_index=False).agg({
    'item_cnt_day': 'sum',
    'item_price': 'mean'
})

## Очистка магазинов

In [9]:
# удаление восклицательного знака в начале магазина
for shop in shops.shop_name:
    if shop.startswith("!"):
        shops['shop_name'] = shops['shop_name'].replace(shop, shop[1:])
    if shop.endswith("фран"):
        shops['shop_name'] = shops['shop_name'].replace(shop, shop[:-5])

print("Повторов: ", shops['shop_name'].duplicated().sum())
print([shop for shop in shops.shop_name])


Повторов:  0
['Якутск Орджоникидзе, 56 фран', 'Якутск ТЦ "Центральный" фран', 'Адыгея ТЦ "Мега"', 'Балашиха ТРК "Октябрь-Киномир"', 'Волжский ТЦ "Волга Молл"', 'Вологда ТРЦ "Мармелад"', 'Воронеж (Плехановская, 13)', 'Воронеж ТРЦ "Максимир"', 'Воронеж ТРЦ Сити-Парк "Град"', 'Выездная Торговля', 'Жуковский ул. Чкалова 39м?', 'Жуковский ул. Чкалова 39м²', 'Интернет-магазин ЧС', 'Казань ТЦ "Бехетле"', 'Казань ТЦ "ПаркХаус" II', 'Калуга ТРЦ "XXI век"', 'Коломна ТЦ "Рио"', 'Красноярск ТЦ "Взлетка Плаза"', 'Красноярск ТЦ "Июнь"', 'Курск ТЦ "Пушкинский"', 'Москва "Распродажа"', 'Москва МТРЦ "Афи Молл"', 'Москва Магазин С21', 'Москва ТК "Буденовский" (пав.А2)', 'Москва ТК "Буденовский" (пав.К7)', 'Москва ТРК "Атриум"', 'Москва ТЦ "Ареал" (Беляево)', 'Москва ТЦ "МЕГА Белая Дача II"', 'Москва ТЦ "МЕГА Теплый Стан" II', 'Москва ТЦ "Новый век" (Новокосино)', 'Москва ТЦ "Перловский"', 'Москва ТЦ "Семеновский"', 'Москва ТЦ "Серебряный Дом"', 'Мытищи ТРК "XL-3"', 'Н.Новгород ТРЦ "РИО"', 'Н.Новгород ТР

In [10]:
franchise_to_main = {
    0: 57,
    1: 58,
}

shops = shops[~shops['shop_id'].isin(list(franchise_to_main.keys()))]
sales = sales[~sales['shop_id'].isin(list(franchise_to_main.keys()))]

shops.head()

,shop_name,shop_id
2,"Адыгея ТЦ ""Мега""",2
3,"Балашиха ТРК ""Октябрь-Киномир""",3
4,"Волжский ТЦ ""Волга Молл""",4
5,"Вологда ТРЦ ""Мармелад""",5
6,"Воронеж (Плехановская, 13)",6


## Работа с датами

In [11]:
sales.iloc[:, [0]].info()

<class 'pandas.DataFrame'>
Index: 2920286 entries, 0 to 2935820
Data columns (total 1 columns):
 #   Column  Dtype
---  ------  -----
 0   date    str  
dtypes: str(1)
memory usage: 72.8 MB


In [12]:
sales["date"] = pd.to_datetime(sales["date"], format='%d.%m.%Y')

sales["year"] = sales["date"].dt.year
sales["month"] = sales["date"].dt.month
sales["day"] = sales["date"].dt.day
sales["weekday"] = sales["date"].dt.weekday
sales["week"] = sales["date"].dt.isocalendar().week

In [13]:
sales.head()

,date,date_block_num,shop_id,item_id,item_cnt_day,item_price,year,month,day,weekday,week
0,2013-01-01,0,2,991,1.0,99.0,2013,1,1,1,1
1,2013-01-01,0,2,1472,1.0,2599.0,2013,1,1,1,1
2,2013-01-01,0,2,1905,1.0,249.0,2013,1,1,1,1
3,2013-01-01,0,2,2920,2.0,599.0,2013,1,1,1,1
4,2013-01-01,0,2,3320,1.0,1999.0,2013,1,1,1,1


## Проверка пропусков

In [14]:
sales.isna().sum()

date              0
date_block_num    0
shop_id           0
item_id           0
item_cnt_day      0
item_price        0
year              0
month             0
day               0
weekday           0
week              0
dtype: int64

## Проверка дубликатов

In [15]:
duplicates_by_key = sales.duplicated(subset=['date', 'shop_id', 'item_id'], keep=False).sum()
print(f"Дубликатов по (date, shop_id, item_id): {duplicates_by_key}")

Дубликатов по (date, shop_id, item_id): 0


In [16]:
sales.duplicated().sum() # полных дубликатов

np.int64(0)

## Работа с некорректными значениями

In [17]:
sales[sales['item_price'] < 0]

,date,date_block_num,shop_id,item_id,item_cnt_day,item_price,year,month,day,weekday,week
1371220,2013-05-15,4,32,2973,1.0,-1.0,2013,5,15,2,20


In [18]:
sales = sales.drop(index=1371220)

In [19]:
sales[sales['item_price'] < sales['item_price'].quantile(0.001)]

,date,date_block_num,shop_id,item_id,item_cnt_day,item_price,year,month,day,weekday,week
741925,2013-06-08,5,58,11865,1.0,0.0700,2013,6,8,5,23
741968,2013-06-08,5,58,20146,4.0,0.0875,2013,6,8,5,23
1009573,2013-06-11,5,6,11864,1.0,0.0700,2013,6,11,1,24
2895310,2013-07-31,6,6,11872,1.0,0.0900,2013,7,31,2,31


In [20]:
# Обрезаем 0.2 % данных снизу и 0.2% данных сверху цены, считаем их за выбросы

sales = sales[sales['item_price'] > sales['item_price'].quantile(0.002)]
sales = sales[sales['item_price'] < sales['item_price'].quantile(0.998)]

In [21]:
# Обрезаем 0.2 % данных сверху спроса, считаем их за выбросы

sales = sales[sales['item_cnt_day'] < sales['item_cnt_day'].quantile(0.998)]

## Очистка категорий

In [22]:
# категория
item_categories["main_category"] = (
    item_categories["item_category_name"]
    .str.split(" - ")
    .str[0]
)

# главная категория

def get_super_category(category):
    
    if category.startswith("Игры"):
        return "Игры"

    elif category.startswith("Игровые консоли"):
        return "Консоли"

    elif category.startswith("Аксессуары"):
        return "Аксессуары"

    elif category.startswith("Книги"):
        return "Книги"

    elif category.startswith("Музыка"):
        return "Музыка"

    elif category.startswith("Кино"):
        return "Кино"

    elif category.startswith("Подарки"):
        return "Подарки"

    elif category.startswith("Программы"):
        return "Программы"

    elif category.startswith("Карты оплаты"):
        return "Карты оплаты"

    elif category.startswith("Чистые носители"):
        return "Чистые носители"

    else:
        return "Прочее"
    
item_categories["super_category"] = (
    item_categories["item_category_name"]
    .apply(get_super_category)
)

item_categories.head()

,item_category_name,item_category_id,main_category,super_category
0,PC - Гарнитуры/Наушники,0,PC,Прочее
1,Аксессуары - PS2,1,Аксессуары,Аксессуары
2,Аксессуары - PS3,2,Аксессуары,Аксессуары
3,Аксессуары - PS4,3,Аксессуары,Аксессуары
4,Аксессуары - PSP,4,Аксессуары,Аксессуары


## Агрегация данных по месяцам

In [24]:
monthly_sales = (
    sales
    .groupby(
        ["date_block_num", "year", "month", "shop_id", "item_id"],
        as_index=False
    )
    .agg(
        item_cnt_month=("item_cnt_day", "sum"),
        avg_price=("item_price", "mean")
    )
)

monthly_sales.head()

,date_block_num,year,month,shop_id,item_id,item_cnt_month,avg_price
0,0,2013,1,2,27,1.0,2499.0
1,0,2013,1,2,33,1.0,499.0
2,0,2013,1,2,317,1.0,299.0
3,0,2013,1,2,438,1.0,299.0
4,0,2013,1,2,471,2.0,399.0


## Группировка

In [26]:
full_sales_info = pd.merge(left= monthly_sales, right= items, 
                           how= 'left', left_on= 'item_id', right_on= 'item_id')\
                    .merge(right= item_categories, 
                           how= 'left', left_on= 'item_category_id', right_on= 'item_category_id')\
                    .merge(right= shops, 
                           how= 'left', left_on= 'shop_id', right_on= 'shop_id')
full_sales_info.head()

,date_block_num,year,month,shop_id,item_id,item_cnt_month,avg_price,item_name,item_category_id,item_category_name,main_category,super_category,shop_name
0,0,2013,1,2,27,1.0,2499.0,"007 Legends [PS3, русская версия]",19,Игры - PS3,Игры,Игры,"Адыгея ТЦ ""Мега"""
1,0,2013,1,2,33,1.0,499.0,1+1 (BD),37,Кино - Blu-Ray,Кино,Кино,"Адыгея ТЦ ""Мега"""
2,0,2013,1,2,317,1.0,299.0,1С:Аудиокниги. Мединский В. Мифы о России. О р...,45,Книги - Аудиокниги 1С,Книги,Книги,"Адыгея ТЦ ""Мега"""
3,0,2013,1,2,438,1.0,299.0,1С:Аудиотеатр. Лучшие произведения русских пис...,45,Книги - Аудиокниги 1С,Книги,Книги,"Адыгея ТЦ ""Мега"""
4,0,2013,1,2,471,2.0,399.0,1С:Бухгалтерия 8 (ред.3.0) как на ладони. Изд ...,49,Книги - Методические материалы 1С,Книги,Книги,"Адыгея ТЦ ""Мега"""


### Проверка после группировки

In [27]:
full_sales_info.isna().sum()

date_block_num        0
year                  0
month                 0
shop_id               0
item_id               0
item_cnt_month        0
avg_price             0
item_name             0
item_category_id      0
item_category_name    0
main_category         0
super_category        0
shop_name             0
dtype: int64

In [28]:
full_sales_info.shape

(1597126, 13)

In [29]:
full_sales_info.describe()

,date_block_num,year,month,shop_id,item_id,item_cnt_month,avg_price,item_category_id
count,1.597126e+06,1.597126e+06,1.597126e+06,1.597126e+06,1.597126e+06,1.597126e+06,1.597126e+06,1.597126e+06
mean,1.472713e+01,2.013796e+03,6.175598e+00,3.297285e+01,1.067595e+04,2.143725e+00,7.603775e+02,4.156457e+01
std,9.510940e+00,7.771055e-01,3.449085e+00,1.641522e+01,6.243279e+03,4.482704e+00,1.234136e+03,1.630409e+01
min,0.000000e+00,2.013000e+03,1.000000e+00,2.000000e+00,0.000000e+00,-2.200000e+01,3.808333e+00,0.000000e+00
25%,6.000000e+00,2.013000e+03,3.000000e+00,2.100000e+01,5.038000e+03,1.000000e+00,1.990000e+02,3.000000e+01
50%,1.400000e+01,2.014000e+03,6.000000e+00,3.100000e+01,1.048700e+04,1.000000e+00,3.990000e+02,4.000000e+01
75%,2.300000e+01,2.014000e+03,9.000000e+00,4.700000e+01,1.606800e+04,2.000000e+00,8.984000e+02,5.500000e+01
max,3.300000e+01,2.015000e+03,1.200000e+01,5.900000e+01,2.216900e+04,2.670000e+02,2.089900e+04,8.300000e+01


In [30]:
full_sales_info.info()

<class 'pandas.DataFrame'>
RangeIndex: 1597126 entries, 0 to 1597125
Data columns (total 13 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   date_block_num      1597126 non-null  int64  
 1   year                1597126 non-null  int32  
 2   month               1597126 non-null  int32  
 3   shop_id             1597126 non-null  int64  
 4   item_id             1597126 non-null  int64  
 5   item_cnt_month      1597126 non-null  float64
 6   avg_price           1597126 non-null  float64
 7   item_name           1597126 non-null  str    
 8   item_category_id    1597126 non-null  int64  
 9   item_category_name  1597126 non-null  str    
 10  main_category       1597126 non-null  object 
 11  super_category      1597126 non-null  str    
 12  shop_name           1597126 non-null  str    
dtypes: float64(2), int32(2), int64(4), object(1), str(4)
memory usage: 366.0+ MB


### Сохранение датасета

In [31]:
full_sales_info.to_parquet(
    "data/interim/monthly_sales.parquet"
)